In [1]:
from scipy.stats import median_abs_deviation

import sys
sys.path.insert(0, '/home/workspace/mm_bm_chip_organoids')
sys.path.insert(0, '/home/workspace/')

from py_util import *
from utilities import *

hdir = '/home/workspace'
wdir = hdir + "/mm_bm_chip_organoids/EXP-01244"
objdir = wdir + "/object_building/objects/"

# adata = sc.read_h5ad(objdir + "processed_merged_adata.h5ad")

### Generating raw adata object

In [2]:
cr_outs_path = wdir + "/cr_outs"

sample_data = {
    'week2': {'id': 'OR07965-01', 'name': 'Week 2'},
    'week3': {'id': 'OR07965-02', 'name': 'Week 3'},
    'week4': {'id': 'OR00001', 'name': 'Week 4'},
    'bm': {'id': 'BMC07965-007', 'name': 'BMMC Start Sample'},
    'msc': {'id': 'CELL00911', 'name': 'MSC Start Sample'}
}

# Find all filtered_feature_bc_matrix.h5 files in the directory structure
h5_paths = [os.path.join(root, 'sample_filtered_feature_bc_matrix.h5') 
           for root, _, files in os.walk(cr_outs_path) 
           if 'sample_filtered_feature_bc_matrix.h5' in files]

samples = [sample['id'] for sample in sample_data.values()]

final_adatas = {}

for sample in samples:

    # Get only the sample h5 paths
    paths = [path for path in h5_paths if sample in path]

    # Dictionary to store AnnData objects for each sample
    adatas = {}
    
    # Process each H5 file
    for path in paths:
        # Extract sample name from path
        name = path.split('per_sample_outs/')[1].split('/')[0]
        
        # Read the H5 file and create AnnData object
        adata = sc.read_10x_h5(path)
        adata.var_names_make_unique()
    
        adata.obs['sample'] = name
    
        adata.obs['base_sample'] = adata.obs['sample'].str.replace(r'_\d+$', '', regex=True)      # Add metadata column for batched replicates

        adata.obs['sample_type'] = adata.obs['base_sample'].replace(
            {data['id']: sample_type for sample_type, data in sample_data.items()}
        )
        
        adata.obs['name'] = adata.obs['sample_type'].replace(
            {sample_type: data['name'] for sample_type, data in sample_data.items()}
        )
        
        adatas[name] = adata.copy()
    
    adata = ad.concat(adatas.values(), join='outer', merge='same')

    final_adatas[adata.obs['sample_type'].unique()[0]] = adata.copy()

adata = ad.concat(final_adatas.values(), join = 'outer', merge = 'same')

adata.write(objdir + 'not_ds_all_samples_raw.h5ad', compression='gzip')

... storing 'sample' as categorical
... storing 'base_sample' as categorical
... storing 'sample_type' as categorical
... storing 'name' as categorical
... storing 'feature_types' as categorical
... storing 'genome' as categorical


In [ ]:
list(adata.obs['sample_type'].unique())